# 📈 InterviewIQ AI — Rigorous Model Evaluation & Metrics
### Notebook 04: Validation, Error Analysis & Diagnostic Plots

This notebook performs formal evaluation on all trained models:
1. Accuracy, Precision, Recall, and F1-Scores across all role categories.
2. Confusion Matrices and Residual Diagnostics for regression models.
3. Cross-Validation stability testing.
4. Identification of failure modes, boundary conditions, and production constraints.

In [ ]:
import os
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    mean_squared_error,
    r2_score,
    mean_absolute_error
)
from sklearn.model_selection import cross_val_score

models_dir = '../models'
data_dir = '../data'

## 1. Evaluating Skill & Role Classification Model

In [ ]:
role_clf = joblib.load(os.path.join(models_dir, 'skill_classifier.joblib'))
role_vec = joblib.load(os.path.join(models_dir, 'skill_vectorizer.joblib'))
role_le = joblib.load(os.path.join(models_dir, 'skill_label_encoder.joblib'))

df_resume = pd.read_csv(os.path.join(data_dir, 'resume_skills.csv'))
X_role_raw = df_resume['resume_text'].fillna('')
y_role_true = role_le.transform(df_resume['role_category'])

X_role_tfidf = role_vec.transform(X_role_raw)
y_role_pred = role_clf.predict(X_role_tfidf)

print("--- Role Classifier Overall Metrics ---")
print(classification_report(y_role_true, y_role_pred, target_names=role_le.classes_))

# 5-fold Cross-validation
cv_scores = cross_val_score(role_clf, X_role_tfidf, y_role_true, cv=5)
print(f"5-Fold Cross Validation Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

## 2. Evaluating Answer Scoring Model (Regression & Quality)

In [ ]:
ans_scorer = joblib.load(os.path.join(models_dir, 'answer_scorer.joblib'))
ans_vec = joblib.load(os.path.join(models_dir, 'answer_vectorizer.joblib'))
ans_qual_clf = joblib.load(os.path.join(models_dir, 'answer_quality_classifier.joblib'))

df_ans = pd.read_csv(os.path.join(data_dir, 'interview_answers.csv'))
combined_ans = df_ans['question'] + ' [SEP] ' + df_ans['answer']
y_score_true = df_ans['score'] * 10.0

X_ans_tfidf = ans_vec.transform(combined_ans)
y_score_pred = ans_scorer.predict(X_ans_tfidf)

residuals = y_score_true - y_score_pred
print(f"Regression Metrics across Full Dataset:")
print(f"  R² Score: {r2_score(y_score_true, y_score_pred):.4f}")
print(f"  RMSE:     {np.sqrt(mean_squared_error(y_score_true, y_score_pred)):.2f}")
print(f"  MAE:      {mean_absolute_error(y_score_true, y_score_pred):.2f}")

## 3. Residual & Prediction Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Actual vs Predicted
axes[0].scatter(y_score_true, y_score_pred, alpha=0.6, color='#4F46E5', edgecolors='none')
axes[0].plot([10, 100], [10, 100], 'r--', lw=2, label='Ideal 1:1')
axes[0].set_title('Predicted vs Actual Scores')
axes[0].set_xlabel('Actual Score')
axes[0].set_ylabel('Predicted Score')
axes[0].legend()

# Residual distribution
axes[1].hist(residuals, bins=20, color='#10B981', edgecolor='white', alpha=0.8)
axes[1].axvline(0, color='red', linestyle='--', lw=2)
axes[1].set_title('Residual Error Distribution (Actual - Predicted)')
axes[1].set_xlabel('Error')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

## 4. Production Limitations & Bias Documentation
1. **Domain Overlap**: Fullstack profiles often share keywords with pure Frontend and Backend resumes.
2. **Extensibility**: TF-IDF models require retraining if novel technical keywords (e.g., emerging frameworks) appear.
3. **Hybrid Architecture Advantage**: InterviewIQ utilizes this baseline ML model alongside an LLM reasoning engine in `llm_service.py`, combining deterministic speed with rich conversational depth.